## Week 36
Familiarize yourself with the dataset card, download the dataset and explore its
columns. Summarize data statistics (size, word count, etc.) for training and
validation data in the languages Arabic (ar), Korean (ko) and Telugu (te).

For each of the languages Arabic, Korean and Telugu, report the 5 most
common words in the questions from the training set and their count, as well
as their English translation. What kind of words are they?

Implement a rule-based classifier that predicts whether a question is an-
swerable or impossible, only using the document (context) and question.

You
may use machine translation as a component. Use the answerable field to
evaluate it on the validation set. What is the performance of your classifier for
each of the languages Arabic, Korean and Telugu?

In [ ]:
from datasets import load_dataset
dataset = load_dataset("coastalcph/tydi_xor_rc")
train_set = dataset["train"]
validation_set = dataset["validation"]

### Week 36

In [ ]:
validation_set = pd.read_parquet("data/validation.parquet")
train_set = pd.read_parquet("data/train.parquet")

In [ ]:
validation_set.shape

(3011, 7)

In [ ]:
train_set.shape

(15343, 7)

In [ ]:
train_set.columns


Index(['question', 'context', 'lang', 'answerable', 'answer_start', 'answer',
       'answer_inlang'],
      dtype='object')

In [ ]:
strip_punctuation = lambda line: [word.strip(string.punctuation+"؟")for word in line.split(" ")]
count_words = lambda line: len(line)

In [ ]:
#!pip install torch

In [ ]:
from transformers import pipeline
import torch

In [ ]:
import torch
print("Torch:", torch.__version__)        # should show +cu126
print("CUDA runtime:", torch.version.cuda) # '12.6'
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from datasets import Dataset
from transformers import pipeline
import pandas as pd
import torch
from itertools import chain


In [ ]:
LANG_CODE = {"ko": "kor_Hang", "ar": "arb_Arab", "te": "tel_Telu"}
device = 0 if torch.cuda.is_available() else -1

translator = pipeline(
    "translation",
    model="facebook/nllb-200-distilled-600M",
    tokenizer="facebook/nllb-200-distilled-600M",
    device=device
)



Device set to use cuda:0


In [ ]:
def hf_translate_questions(lang_df, l, batch_size: int = 32):
    work_df = lang_df[["question"]].copy()
    work_df["question"] = work_df["question"].fillna("").astype(str)

    ds = Dataset.from_pandas(work_df.reset_index(drop=True), preserve_index=False)

    def _translate_batch(batch):
        outs = translator(
            batch["question"],
            src_lang=LANG_CODE[l],
            tgt_lang="eng_Latn",
            max_length=256,
            truncation=True,
        )
        return {"question_translated": [o["translation_text"] for o in outs]}

    ds_out = ds.map(_translate_batch, batched=True, batch_size=batch_size)

    translated_df = ds_out.to_pandas()
    return translated_df[["question_translated"]]



In [ ]:
def safe_top_tokens(lang_df, l, topn: int = 5):
    col = "question_stripped"
    if lang_df[col].dtype == object and lang_df[col].map(lambda x: isinstance(x, list)).any():
        tokens = list(chain.from_iterable(x or [] for x in lang_df[col].tolist()))
    else:
        tokens = list(chain.from_iterable((str(x or "")).split() for x in lang_df[col].tolist()))
    tokens = [t.strip() for t in tokens if isinstance(t, str) and t.strip()]
    counts = pd.Series(tokens).value_counts().head(topn)

    def translate_top_words(counts_series, lcode):
        tx = [
            translator(w, src_lang=LANG_CODE[lcode], tgt_lang="eng_Latn", max_length=64)[0]["translation_text"]
            for w in counts_series.index
        ]
        out = counts_series.reset_index()
        out.columns = ["token", "count"]
        out["translation"] = tx
        return out

    return translate_top_words(counts, l)


In [ ]:

def statistics(df, col=["question", "context"], translate = False):
    langs = ["ko", "ar", "te"]
    df = df[df["lang"].isin(langs)].copy()

    for c in col:
        df[f"{c}_stripped"] = df[c].apply(strip_punctuation)
        df[f"{c}_wordcount"] = df[f"{c}_stripped"].apply(count_words)

    print(
        df.groupby(["lang", "answerable"]).agg(
            question_wordcount_mean=("question_wordcount", "mean"),
            context_wordcount_mean=("context_wordcount", "mean"),
            question_wordcount_sum=("question_wordcount", "sum"),
            context_wordcount_sum=("context_wordcount", "sum"),
            question_wordcount_max=("question_wordcount", "max"),
            question_wordcount_min=("question_wordcount", "min"),
            count=("question_wordcount", "count")
        ).round(0)
    )

    if translate:
        df["question_translated"] = None

    for l in langs:
        mask = df["lang"] == l
        lang_df = df.loc[mask].copy()

        print(f"\nTop tokens for {l} -> en:")
        top_counts = safe_top_tokens(lang_df, l)
        print(top_counts)

        if translate:
            translated_df = hf_translate_questions(lang_df, l, batch_size=32)
            df.loc[mask, "question_translated"] = translated_df["question_translated"].values
            torch.cuda.empty_cache()

    return df

In [ ]:
translate = False

In [ ]:
val_for_stat = statistics(validation_set, translate=translate)


                 question_wordcount_mean  context_wordcount_mean  \
lang answerable                                                    
ar   False                           8.0                   112.0   
     True                            7.0                   103.0   
ko   False                           5.0                   108.0   
     True                            5.0                    95.0   
te   False                           6.0                   112.0   
     True                            6.0                   105.0   

                 question_wordcount_sum  context_wordcount_sum  \
lang answerable                                                  
ar   False                          406                   5813   
     True                          2436                  37285   
ko   False                           95                   2051   
     True                          1636                  32124   
te   False                          575                  10

In [ ]:
train_for_stat = statistics(train_set, translate=translate)

                 question_wordcount_mean  context_wordcount_mean  \
lang answerable                                                    
ar   False                           8.0                   124.0   
     True                            7.0                   102.0   
ko   False                           5.0                   104.0   
     True                            5.0                    97.0   
te   False                           6.0                   118.0   
     True                            6.0                    87.0   

                 question_wordcount_sum  context_wordcount_sum  \
lang answerable                                                  
ar   False                         1948                  31580   
     True                         15509                 234189   
ko   False                          325                   6574   
     True                         11524                 228653   
te   False                          277                   5

In [ ]:
if translate:
    val_for_stat.to_csv("data/validation_translated.csv")
    train_for_stat.to_csv("data/train_translated.csv")

  I wanted to strip punctuation, but certain languages that we do not include in the analysis have special punctuation that has to be included in thw stripping pool

### Week 37 - rule-based classifier and lstm language modelling


In [3]:
import pandas as pd
import string
import numpy as np

In [4]:
val_tr = pd.read_csv("data/validation_translated.csv")
train_tr = pd.read_csv("data/train_translated.csv")

In [5]:
val_tr.shape

(1155, 13)

In [6]:
train_tr.shape

(6335, 13)

In [7]:
train_tr

,Unnamed: 0,question,context,lang,answerable,answer_start,answer,answer_inlang,question_stripped,question_wordcount,context_stripped,context_wordcount,question_translated
0,4792,30년 전쟁의 승자는 누구인가?,The conflict between France and Spain continue...,ko,True,21,France,NaN,"['30년', '전쟁의', '승자는', '누구인가']",4,"['The', 'conflict', 'between', 'France', 'and'...",108,Who is the winner of the Thirty Years' War?
1,4793,엑스선은 누가 발견하였는가?,"X-rays make up X-radiation, a form of electrom...",ko,True,503,Wilhelm Röntgen,NaN,"['엑스선은', '누가', '발견하였는가']",3,"['X-rays', 'make', 'up', 'X-radiation', 'a', '...",122,Who discovered X-rays?
2,4794,아테네에서 언제 가장 최근의 올림픽이 올렸나요?,"In 2022, Beijing will become the first-ever ci...",ko,True,188,2004,NaN,"['아테네에서', '언제', '가장', '최근의', '올림픽이', '올렸나요']",6,"['In', '2022', 'Beijing', 'will', 'become', 't...",197,When was the last Olympic Games held in Athens?
3,4795,세상에서 가장 오래된 방송사는 무엇인가?,The British Broadcasting Corporation (BBC) is ...,ko,True,4,British Broadcasting Corporation (BBC),NaN,"['세상에서', '가장', '오래된', '방송사는', '무엇인가']",5,"['The', 'British', 'Broadcasting', 'Corporatio...",70,What's the oldest broadcaster in the world?
4,4796,팔레스타인 수도는 어딘가요?,"Palestine ( '), officially the State of Palest...",ko,True,205,Jerusalem,NaN,"['팔레스타인', '수도는', '어딘가요']",3,"['Palestine', '', '', 'officially', 'the', 'St...",84,Where's the Palestinian capital?
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6330,15338,소말리아는 2차 개헌을 언제 했나요?,"In February 2012, Somali government officials ...",ko,True,923,23 June 2012,NaN,"['소말리아는', '2차', '개헌을', '언제', '했나요']",5,"['In', 'February', '2012', 'Somali', 'governme...",181,When did Somalia make its second inauguration?
6331,15339,세상에서 가장 먼저 시작된 교통수단은 무엇인가?,The first earth tracks were created by humans ...,ko,True,160,animals,NaN,"['세상에서', '가장', '먼저', '시작된', '교통수단은', '무엇인가']",6,"['The', 'first', 'earth', 'tracks', 'were', 'c...",118,What was the world's first transportation system?
6332,15340,2019년 이집트의 지도자는 누구인가?,"Abdel Fattah Saeed Hussein Khalil El-Sisi ( """"...",ko,True,0,Abdel Fattah Saeed Hussein Khalil El-Sisi,NaN,"['2019년', '이집트의', '지도자는', '누구인가']",4,"['Abdel', 'Fattah', 'Saeed', 'Hussein', 'Khali...",30,Who is Egypt's leader in 2019?
6333,15341,독일에서 가장 인구밀도가 높은 도시는 무엇인가?,Munich (; ; ) is the capital and most populous...,ko,True,205,Berlin,NaN,"['독일에서', '가장', '인구밀도가', '높은', '도시는', '무엇인가']",6,"['Munich', '', '', '', 'is', 'the', 'capital',...",118,What is the most densely populated city in Ger...


In [8]:
strip_first = lambda x: x.split()[0]

In [9]:
import re

question_contractions = {
    "what's": "what is",
    "where's": "where is",
    "who's": "who is",
    "when's": "when is",
    "why's": "why is",
    "how's": "how is",
    "in what year": "when",
    "in what month": "when",
    "in what day": "when",
    "in what hour": "when",
    "in what minute": "when",
    "in what second": "when",
    "what're": "what are",
    "where're": "where are",
    "who're": "who are",
    "when're": "when are",
    "why're": "why are",
    "how're": "how are",
    "what've": "what have",
    "where've": "where have",
    "who've": "who have",
    "when've": "when have",
    "why've": "why have",
    "how've": "how have",
    "in which country": "where",
    "in which city": "where",
    "in which state": "where",
    "in which region": "where",
    "in which department": "where",
    "in what country": "where",
    "in what city": "where",
    "in what state": "where",
    "in what region": "where",
    "in what department": "where",
    "in what company": "where",
    "in what years": "when",
    "in what months": "when",
    "in what days": "when",
    "in what hours": "when",
    "in what minutes": "when",
    "in what seconds": "when",
    "in which year": "when",
    "in which month": "when",
    "in which day": "when",
    "in which hour": "when",
    "in which minute": "when",
    "in which second": "when",
    "in which direction": "where",
    "on what day": "when",
    "on what date": "when",
    "on what time": "when",

}

def expand_contractions(text, contractions=question_contractions):
    pattern = re.compile(r'\b(' + '|'.join(re.escape(k) for k in contractions.keys()) + r')\b', flags=re.IGNORECASE)
    return pattern.sub(lambda x: contractions[x.group().lower()], text)


In [10]:
train_tr["question_start"] = train_tr["question_translated"].apply(expand_contractions).apply(strip_first).apply(str.lower)
val_tr["question_start"] = val_tr["question_translated"].apply(expand_contractions).apply(strip_first).apply(str.lower)

#### Rule-based

In [11]:
def top_10(df):
    counts = (
        df.groupby(["answerable", "question_start"])
        .size()
        .rename("count")
        .reset_index()
    )

    counts["answerable"] = counts["answerable"].astype(bool)

    top10 = (
        counts.sort_values(["answerable", "count"], ascending=[True, False])\
              .groupby("answerable", group_keys=False)\
              .apply(lambda x: x) \
              .reset_index(drop=True) # Convert back to DataFrame
    )

    opposite_counts = counts.copy()
    opposite_counts["answerable"] = ~opposite_counts["answerable"]
    opposite_counts = opposite_counts.rename(columns={"count": "count_in_opposite_class"})

    top10_with_opposite = (
        top10.merge(
            opposite_counts[["answerable", "question_start", "count_in_opposite_class"]],
            on=["answerable", "question_start"],
            how="left",
        )
        .fillna({"count_in_opposite_class": 0})
    )
    top10_with_opposite["count_in_opposite_class"] = (
        top10_with_opposite["count_in_opposite_class"].astype(int)
    )

    top10_with_opposite["valid"] = np.select(
        [
            top10_with_opposite["count"] > top10_with_opposite["count_in_opposite_class"],
            top10_with_opposite["count"] < top10_with_opposite["count_in_opposite_class"],
        ],
        [True, False],
        default="tie",
    )
    return top10_with_opposite
# top10_with_opposite["is_more_in_answerable"] = np.where(
#     top10_with_opposite["count"] == top10_with_opposite["count_in_opposite_class"],
#     np.nan,
#     top10_with_opposite["count"] > top10_with_opposite["count_in_opposite_class"],
# )



In [12]:
valid = top_10(val_tr)
train = top_10(train_tr)

C:\Users\dnedi\AppData\Local\Temp\ipykernel_11072\1919682493.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  counts.sort_values(["answerable", "count"], ascending=[True, False])\
C:\Users\dnedi\AppData\Local\Temp\ipykernel_11072\1919682493.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  counts.sort_values(["answerable", "count"], ascending=[True, False])\


In [13]:
train[train["answerable"]==False].head(6).to_latex()

'\\begin{tabular}{lrlrrl}\n\\toprule\n & answerable & question_start & count & count_in_opposite_class & valid \\\\\n\\midrule\n0 & False & is & 139 & 61 & True \\\\\n1 & False & can & 37 & 7 & True \\\\\n2 & False & does & 37 & 25 & True \\\\\n3 & False & what & 29 & 1946 & False \\\\\n4 & False & are & 22 & 9 & True \\\\\n5 & False & do & 17 & 6 & True \\\\\n\\bottomrule\n\\end{tabular}\n'

In [14]:
train[train["answerable"]==True].head().to_latex()

'\\begin{tabular}{lrlrrl}\n\\toprule\n & answerable & question_start & count & count_in_opposite_class & valid \\\\\n\\midrule\n24 & True & what & 1946 & 29 & True \\\\\n25 & True & when & 1247 & 4 & True \\\\\n26 & True & who & 1050 & 10 & True \\\\\n27 & True & how & 787 & 7 & True \\\\\n28 & True & where & 516 & 4 & True \\\\\n\\bottomrule\n\\end{tabular}\n'

In [15]:
train_start_nonans = train[(train["valid"] == "True") & (train["answerable"] == False)]["question_start"].tolist()

In [16]:
train_start_nonans

['is',
 'can',
 'does',
 'are',
 'do',
 'has',
 'did',
 'was',
 'could',
 'at',
 'have',
 'were',
 'jack',
 'major,']

In [17]:
def rule_based(df, non_questions):
  df["prediction_answarable"] = np.where(df["question_start"].isin(non_questions), False, True)
  return df

In [18]:
train_pred = rule_based(train_tr, train_start_nonans)
val_pred = rule_based(val_tr, train_start_nonans)

In [19]:
train_pred["accurate"] = train_pred["answerable"] == train_pred["prediction_answarable"]
val_pred["accurate"] = val_pred["answerable"] == val_pred["prediction_answarable"]

In [23]:
def metrics_with_bacc(y_true, y_pred, name="set"):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    # Confusion matrix components (binary, positive class = 1)
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())

    # Safe div helper
    div = lambda a, b: (a / b) if b else 0.0

    accuracy  = div(tp + tn, tp + tn + fp + fn)
    precision = div(tp, tp + fp)
    recall    = div(tp, tp + fn)
    f1        = div(2 * precision * recall, precision + recall)

    tpr_pos = recall
    tpr_neg = div(tn, tn + fp)
    balanced_accuracy = 0.5 * (tpr_pos + tpr_neg)

    print(f"{name} accuracy: {accuracy:.4f},f1: {f1:.4f},baac: {balanced_accuracy:.4f}")
    # print(f"{name} precision: {precision:.4f}")
    # print(f"{name} recall: {recall:.4f}")

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "balanced_accuracy": balanced_accuracy,
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
    }

def accuracy_and_f1(series, name = "sets"):
    accuracy = series.mean()
    tp = series.sum()
    fp = (series == False).sum()
    fn = (series == True).sum() - tp
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    f1 = 2 * precision * recall / (precision + recall)
    print(f"{name} accuracy: {accuracy:.4f}, f1: {f1:.4f}")
    #print(f"{name} precision: {precision:.4f}")
    #print(f"{name} recall: {recall:.4f}")


In [24]:
LANG = {"ar": "Arabic", "ko": "Korean", "te": "Telugu"}

In [25]:
for l in LANG.keys():
    metrics_with_bacc(train_pred[train_pred["lang"]==l]["answerable"],train_pred[train_pred["lang"]==l]["prediction_answarable"], f"Train data {LANG.get(l)}")
    metrics_with_bacc(val_pred[val_pred["lang"]==l]["answerable"],val_pred[val_pred["lang"]==l]["prediction_answarable"], f"Validation data {LANG.get(l)}")

metrics_with_bacc(train_pred["answerable"],train_pred["prediction_answarable"], "Train data")
metrics_with_bacc(val_pred["answerable"],val_pred["prediction_answarable"], "Validation data")

Train data Arabic accuracy: 0.9644,f1: 0.9799,baac: 0.9646
Validation data Arabic accuracy: 0.9807,f1: 0.9889,baac: 0.9725
Train data Korean accuracy: 0.9756,f1: 0.9874,baac: 0.9180
Validation data Korean accuracy: 0.9635,f1: 0.9805,baac: 0.9062
Train data Telugu accuracy: 0.9675,f1: 0.9835,baac: 0.5218
Validation data Telugu accuracy: 0.7500,f1: 0.8567,baac: 0.4985
Train data accuracy: 0.9694,f1: 0.9837,baac: 0.9048
Validation data accuracy: 0.8987,f1: 0.9432,baac: 0.6942


{'accuracy': 0.8987012987012987,
 'precision': 0.9091760299625468,
 'recall': 0.9798183652875883,
 'f1': 0.9431762991743563,
 'balanced_accuracy': 0.6941774753267209,
 'tp': 971,
 'tn': 67,
 'fp': 97,
 'fn': 20}

In [85]:
valid[(valid["valid"] == "True") & (valid["answerable"] == False)]

,answerable,question_start,count,count_in_opposite_class,valid
1,False,is,29,12,True
4,False,can,9,0,True
5,False,does,9,1,True
6,False,are,7,1,True
9,False,"bharat,",5,0,True
10,False,do,5,0,True
13,False,as,3,2,True
14,False,has,3,0,True
16,False,was,2,0,True


In [65]:
valid[valid["valid"]=="True"]

,answerable,question_start,count,count_in_opposite_class,valid
1,False,Is,29,12,True
4,False,Can,9,0,True
5,False,Does,9,1,True
6,False,Are,7,1,True
9,False,"Bharat,",5,0,True
10,False,Do,5,0,True
13,False,As,3,2,True
14,False,Has,3,0,True
16,False,Was,2,0,True
19,True,What,304,35,True


In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "PrimeQA/tydiqa-boolean-question-classifier"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/264 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/822 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/712M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/711M [00:00<?, ?B/s]

In [ ]:
model.eval()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(119547, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=Fals

In [ ]:

@torch.no_grad()
def get_embeddings_and_tokens(
    series: pd.Series,
    pooling: str = "cls",
    batch_size: int = 16,
    max_length: int = 512,
) -> pd.DataFrame:
    """
    Returns a DataFrame (same index as `series`) with:
      - 'embedding'        : np.ndarray of shape (hidden_dim,)
      - 'tokens'           : list[str] (wordpiece tokens, specials removed)
      - 'tokenized_text'   : str       (tokens joined by space)
    """
    if pooling not in {"cls", "mean"}:
        raise ValueError("pooling must be 'cls' or 'mean'")

    all_idx = []
    emb_chunks = []
    tokens_list = []
    tokenized_text_list = []

    special_ids = set(tokenizer.all_special_ids)

    for start in tqdm(range(0, len(series), batch_size)):
        batch = series.iloc[start:start + batch_size]
        if batch.empty:
            continue
        all_idx.extend(batch.index.tolist())

        enc = tokenizer(
            batch.tolist(),
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
        ).to(device)

        outputs = model(**enc)
        hidden = outputs.last_hidden_state  # [B, T, H]

        if pooling == "cls":
            emb = hidden[:, 0, :]
        else:
            mask = enc["attention_mask"].unsqueeze(-1).float()
            emb = (hidden * mask).sum(dim=1) / mask.sum(dim=1)

        emb_chunks.append(emb.detach().cpu())

        # Build tokens per example (trim to true length, drop special tokens)
        input_ids = enc["input_ids"].detach().cpu()
        attn = enc["attention_mask"].detach().cpu()
        for ids, m in zip(input_ids, attn):
            L = int(m.sum().item())
            ids_trim = ids[:L].tolist()
            # remove specials
            ids_trim = [i for i in ids_trim if i not in special_ids]
            toks = tokenizer.convert_ids_to_tokens(ids_trim)
            tokens_list.append(toks)
            tokenized_text_list.append(" ".join(toks))

        if device.type == "cuda":
            torch.cuda.empty_cache()

    embs = torch.cat(emb_chunks, dim=0).numpy() if emb_chunks else np.empty((0,))
    out = pd.DataFrame(index=pd.Index(all_idx, name=series.index.name))
    out["embedding"] = list(embs)
    out["tokens"] = tokens_list
    out["tokenized_text"] = tokenized_text_list

    return out.reindex(series.index)

def tokenizer_embeddings(
    df: pd.DataFrame,
    cols=("question", "context"),
    pooling: str = "cls",
    batch_size: int = 32,
    max_length: int = 512,
) -> pd.DataFrame:
    """
    For each source column in `cols`, appends:
      - f"{col}_embedding"
      - f"{col}_tokens"
      - f"{col}_tokenized_text"
    """
    out_df = df.copy()
    for c in cols:
        if c not in out_df.columns:
            continue
        print(f"Generating embeddings & tokens for '{c}'...")
        pair_df = get_embeddings_and_tokens(
            out_df[c], pooling=pooling, batch_size=batch_size, max_length=max_length
        ).rename(columns={
            "embedding": f"{c}_embedding",
            "tokens": f"{c}_tokens",
            "tokenized_text": f"{c}_tokenized_text",
        })
        out_df = out_df.join(pair_df, how="left")
    return out_df

In [ ]:
# ---- Example ----
df = pd.DataFrame({"question": ["Is it raining?"], "context": ["It is cloudy in Copenhagen."]})
df_out = tokenizer_embeddings(df)
print(df_out.columns)
print(df_out.loc[df_out.index[0], ["question_tokens", "question_tokenized_text"]])

Generating embeddings & tokens for 'question'...


100%|██████████| 1/1 [00:00<00:00,  2.29it/s]


Generating embeddings & tokens for 'context'...


100%|██████████| 1/1 [00:00<00:00, 18.10it/s]

Index(['question', 'context', 'question_embedding', 'question_tokens',
       'question_tokenized_text', 'context_embedding', 'context_tokens',
       'context_tokenized_text'],
      dtype='object')
question_tokens            [Is, it, rain, ##ing, ?]
question_tokenized_text          Is it rain ##ing ?
Name: 0, dtype: object


In [ ]:
train_tr_tok = tokenizer_embeddings(train_tr)
val_tr_tok = tokenizer_embeddings(val_tr)


Generating embeddings & tokens for 'question'...


  3%|▎         | 5/198 [00:00<00:13, 14.51it/s]

model.safetensors:   0%|          | 0.00/711M [00:00<?, ?B/s]

100%|██████████| 198/198 [00:11<00:00, 17.25it/s]


Generating embeddings & tokens for 'context'...


100%|██████████| 198/198 [02:25<00:00,  1.36it/s]


Generating embeddings & tokens for 'question'...


100%|██████████| 37/37 [00:02<00:00, 16.92it/s]


Generating embeddings & tokens for 'context'...


100%|██████████| 37/37 [00:26<00:00,  1.39it/s]


In [ ]:
train_tr_tok.columns

Index(['Unnamed: 0', 'question', 'context', 'lang', 'answerable',
       'answer_start', 'answer', 'answer_inlang', 'question_stripped',
       'question_wordcount', 'context_stripped', 'context_wordcount',
       'question_translated', 'question_start', 'question_embedding',
       'question_tokens', 'question_tokenized_text', 'context_embedding',
       'context_tokens', 'context_tokenized_text'],
      dtype='object')

In [ ]:
train_tr_tok.head(5)

,Unnamed: 0,question,context,lang,answerable,answer_start,answer,answer_inlang,question_stripped,question_wordcount,context_stripped,context_wordcount,question_translated,question_start,question_embedding,question_tokens,question_tokenized_text,context_embedding,context_tokens,context_tokenized_text
0,4792,30년 전쟁의 승자는 누구인가?,The conflict between France and Spain continue...,ko,True,21,France,NaN,"['30년', '전쟁의', '승자는', '누구인가']",4,"['The', 'conflict', 'between', 'France', 'and'...",108,Who is the winner of the Thirty Years' War?,Who,"[1.6347834, -0.98751277, 0.097479045, 0.228975...","[30, ##년, 전쟁, ##의, 승, ##자는, 누, ##구, ##인, ##가, ?]",30 ##년 전쟁 ##의 승 ##자는 누 ##구 ##인 ##가 ?,"[-0.7733206, 0.8051937, 0.41695708, -0.0355528...","[The, conflict, between, France, and, Spain, c...",The conflict between France and Spain continue...
1,4793,엑스선은 누가 발견하였는가?,"X-rays make up X-radiation, a form of electrom...",ko,True,503,Wilhelm Röntgen,NaN,"['엑스선은', '누가', '발견하였는가']",3,"['X-rays', 'make', 'up', 'X-radiation', 'a', '...",122,Who discovered X-rays?,Who,"[1.6302526, -0.9880773, 0.09621535, 0.23146597...","[엑, ##스, ##선, ##은, 누, ##가, 발, ##견, ##하, ##였, #...",엑 ##스 ##선 ##은 누 ##가 발 ##견 ##하 ##였 ##는 ##가 ?,"[-0.7976921, 0.7418729, 0.3573977, -0.06668669...","[X, -, ray, ##s, make, up, X, -, radiation, ,,...","X - ray ##s make up X - radiation , a form of ..."
2,4794,아테네에서 언제 가장 최근의 올림픽이 올렸나요?,"In 2022, Beijing will become the first-ever ci...",ko,True,188,2004,NaN,"['아테네에서', '언제', '가장', '최근의', '올림픽이', '올렸나요']",6,"['In', '2022', 'Beijing', 'will', 'become', 't...",197,When was the last Olympic Games held in Athens?,When,"[1.6525371, -0.99947554, 0.08875036, 0.2332689...","[아, ##테, ##네, ##에서, 언, ##제, 가장, 최, ##근, ##의, 올...",아 ##테 ##네 ##에서 언 ##제 가장 최 ##근 ##의 올림픽 ##이 올 ##...,"[-0.7564159, 0.8167616, 0.50251335, -0.0409449...","[In, 2022, ,, Beijing, will, become, the, firs...","In 2022 , Beijing will become the first - ever..."
3,4795,세상에서 가장 오래된 방송사는 무엇인가?,The British Broadcasting Corporation (BBC) is ...,ko,True,4,British Broadcasting Corporation (BBC),NaN,"['세상에서', '가장', '오래된', '방송사는', '무엇인가']",5,"['The', 'British', 'Broadcasting', 'Corporatio...",70,What's the oldest broadcaster in the world?,what,"[1.6331843, -0.98917705, 0.09663707, 0.2301011...","[세, ##상, ##에서, 가장, 오, ##래, ##된, 방송, ##사는, 무, #...",세 ##상 ##에서 가장 오 ##래 ##된 방송 ##사는 무 ##엇 ##인 ##가 ?,"[-0.7746885, 0.806728, 0.44596887, -0.02970119...","[The, British, Broadcasting, Corporation, (, B...",The British Broadcasting Corporation ( BBC ) i...
4,4796,팔레스타인 수도는 어딘가요?,"Palestine ( '), officially the State of Palest...",ko,True,205,Jerusalem,NaN,"['팔레스타인', '수도는', '어딘가요']",3,"['Palestine', '', '', 'officially', 'the', 'St...",84,Where's the Palestinian capital?,where,"[1.6366284, -0.98907083, 0.09580283, 0.2281950...","[팔, ##레스, ##타, ##인, 수도, ##는, 어, ##딘, ##가, ##요, ?]",팔 ##레스 ##타 ##인 수도 ##는 어 ##딘 ##가 ##요 ?,"[-0.7477749, 0.8391133, 0.5210368, -0.03393136...","[Palestine, (, ', ), ,, officially, the, State...","Palestine ( ' ) , officially the State of Pale..."


In [ ]:
train_tr_tok.to_csv("data/train_tr_tok.csv")
val_tr_tok.to_csv("data/val_tr_tok.csv")

In [ ]:
!pip install -q torchtext

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 50.0 MB/s eta 0:00:00


In [ ]:
print(torch.__version__)

2.8.0+cu126


In [ ]:
import torch.nn as nn

In [ ]:
from datasets import Dataset

def data_loader(train, val, tokenizer, col = "context", lang = None, max_length=128):
    x_col = col
    y_col = "answerable"
    if lang is not None:
        train = train[train["lang"] == lang].copy()
        val   = val[val["lang"] == lang].copy()


    train = train[[x_col, y_col]].dropna().copy()
    val   = val[[x_col, y_col]].dropna().copy()
    train[y_col] = train[y_col].astype(int)
    val[y_col]   = val[y_col].astype(int)

    # convert pandas to HF Dataset
    train_hf = Dataset.from_pandas(train[[x_col, y_col]])
    val_hf = Dataset.from_pandas(val[[x_col, y_col]])

    def preprocess(examples):
        enc = tokenizer(examples[x_col], truncation=True, padding="max_length", max_length=max_length)
        enc["labels"] = examples[y_col]
        return enc

    # map + clean columns
    train_enc = train_hf.map(preprocess, batched=True, remove_columns=[x_col, y_col])
    val_enc   = val_hf.map(preprocess,   batched=True, remove_columns=[x_col, y_col])

    # make PyTorch-ready
    train_enc.set_format("torch")
    val_enc.set_format("torch")
    return train_enc, val_enc


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

In [ ]:
tr_ko,val_ko = data_loader(train_tr, val_tr,tokenizer, col="question", lang="ko")
tr_te,val_te = data_loader(train_tr, val_tr, tokenizer, col="question", lang="te")
tr_ar,val_ar = data_loader(train_tr, val_tr, tokenizer, col="question", lang="ar")
tr_con, val_con = data_loader(train_tr, val_tr, tokenizer, col="context", lang=None)

Map:   0%|          | 0/2422 [00:00<?, ? examples/s]

Map:   0%|          | 0/356 [00:00<?, ? examples/s]

Map:   0%|          | 0/1355 [00:00<?, ? examples/s]

Map:   0%|          | 0/384 [00:00<?, ? examples/s]

Map:   0%|          | 0/2558 [00:00<?, ? examples/s]

Map:   0%|          | 0/415 [00:00<?, ? examples/s]

Map:   0%|          | 0/6335 [00:00<?, ? examples/s]

Map:   0%|          | 0/1155 [00:00<?, ? examples/s]

Creating CSV from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating CSV from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating CSV from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating CSV from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

Creating CSV from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

1600717

You can load the saved datasets from disk using the `load_from_disk` function:

### Week 37 - data only


In [ ]:
!huggingface-cli login


⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    A token is already saved on your machine. Run `hf auth whoami` to get more information or `hf auth logout` if you want to log out.
    Setting a new token will erase the existing one.
    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add t

In [ ]:
from datasets import load_dataset

# Define the repository ID
repo_id = "Meduzka/TYDI_qa_languages"

# Load the train and validation datasets for each language and the context
try:
    train_ko = load_dataset(repo_id, split="train", data_files="train_ko.csv")
    val_ko = load_dataset(repo_id, split="train", data_files="val_ko.csv")

    train_te = load_dataset(repo_id, split="train", data_files="train_te.csv")
    val_te = load_dataset(repo_id, split="train", data_files="val_te.csv")

    train_ar = load_dataset(repo_id, split="train", data_files="train_ar.csv")
    val_ar = load_dataset(repo_id, split="train", data_files="val_ar.csv")

    train_con = load_dataset(repo_id, split="train", data_files="train_con.csv")
    val_con = load_dataset(repo_id, split="train", data_files="val_con.csv")

    print("Datasets loaded successfully:")
    print("train_ko:", train_ko)
    print("val_ko:", val_ko)
    print("train_te:", train_te)
    print("val_te:", val_te)
    print("train_ar:", train_ar)
    print("val_ar:", val_ar)
    print("train_con:", train_con)
    print("val_con:", val_con)

except Exception as e:
    print(f"Error loading datasets: {e}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Datasets loaded successfully:
train_ko: Dataset({
    features: ['__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 2422
})
val_ko: Dataset({
    features: ['__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 356
})
train_te: Dataset({
    features: ['__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 1355
})
val_te: Dataset({
    features: ['__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 384
})
train_ar: Dataset({
    features: ['__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 2558
})
val_ar: Dataset({
    features: ['__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 415
})
train_con: Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 6335
})
val_con: Dataset({
    features: 

In [ ]:
import torch
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, layer_dim, pad_id=0, dropout=0.0, ff_dim=None):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim
        ff_dim = ff_dim or hidden_dim  # intermediate size for the MLP head

        # token IDs -> embeddings
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)

        # stacked LSTM
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=layer_dim,
            batch_first=True,
            dropout=dropout if layer_dim > 1 else 0.0
        )


        # MLP head: Linear -> ReLU -> Linear -> logits
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, ff_dim),
            nn.ReLU(inplace=True),
            # nn.Linear(ff_dim, ff_dim)
            # nn.ReLU(inplace=True),
            nn.Linear(ff_dim, vocab_size)
        )

    def forward(self, x, h0=None, c0=None):
        """
        x:  (B, T) token IDs (Long)
        h0/c0: (num_layers, B, hidden_dim) or None
        returns:
            logits: (B, T, vocab_size)
            hn, cn: final states
        """
        emb = self.embedding(x)  # (B, T, E)

        if h0 is None or c0 is None:
            h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim, device=x.device)
            c0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim, device=x.device)

        out, (hn, cn) = self.lstm1(emb, (h0, c0))  # (B, T, H)
        logits = self.head(out)                    # (B, T, V)
        return logits, hn, cn


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import balanced_accuracy_score

def trainer(df, tokenizer, num_epochs=100, use_labels=False):
    df.set_format("torch", columns=["input_ids"] + (["labels"] if "labels" in df.features else []))

    batch_size = 64
    train_loader = DataLoader(df, batch_size=batch_size, shuffle=True, drop_last=False)

    vocab_size = tokenizer.vocab_size
    embed_dim, hidden_dim, layer_dim = 64, 128, 5
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0

    model = LSTMModel(vocab_size, embed_dim, hidden_dim, layer_dim, pad_id=pad_id)

    criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Using device:", device)
    model.to(device)

    for epoch in range(num_epochs):
        model.train()
        epoch_loss, n_batches = 0.0, 0
        y_true_all, y_pred_all = [], []  # for balanced accuracy

        for batch in train_loader:
            input_ids = batch["input_ids"].to(device).long()
            optimizer.zero_grad(set_to_none=True)

            logits, _, _ = model(input_ids)  # [B,T,V]

            if use_labels and "labels" in batch:
                labels = batch["labels"].to(device).long()      # [B,T]
                logits_aligned = logits                          # [B,T,V]
            else:
                labels = input_ids[:, 1:]                        # [B,T-1]
                logits_aligned = logits[:, :-1, :]               # [B,T-1,V]

            loss = criterion(logits_aligned.reshape(-1, vocab_size),
                             labels.reshape(-1))
            loss.backward()
            optimizer.step()

            # collect preds/targets (ignore PAD) for balanced accuracy
            with torch.no_grad():
                preds = logits_aligned.argmax(dim=-1)            # [B,T or T-1]
                mask = labels != pad_id
                if mask.any():
                    y_true_all.extend(labels[mask].view(-1).detach().cpu().tolist())
                    y_pred_all.extend(preds[mask].view(-1).detach().cpu().tolist())

            epoch_loss += loss.item()
            n_batches += 1

            del logits, loss, input_ids, logits_aligned, labels
            if device == "cuda":
                torch.cuda.empty_cache()
                torch.cuda.ipc_collect()

        bal_acc = balanced_accuracy_score(y_true_all, y_pred_all) if y_true_all else float("nan")
        print(f"Epoch [{epoch+1}/{num_epochs}] - loss: {epoch_loss/max(1,n_batches):.4f} | balanced_acc: {bal_acc:.4f}")

    return model


In [ ]:
torch.cuda.empty_cache()

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "PrimeQA/tydiqa-boolean-question-classifier"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
model_ko = trainer(train_ko, tokenizer,num_epochs = 10)

Using device: cuda


ValueError: type of [  101  9283 23466 31401 15184 10459  9420 68828  9546 48446 12030 11287
 48549   136   102     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0] unknown: <class 'str'>. Should be one of a python, numpy, pytorch or tensorflow object.

In [ ]:
!zip -r /content/data.zip /content/data

updating: content/data/ (stored 0%)
updating: content/data/arabic/ (stored 0%)
updating: content/data/arabic/val/ (stored 0%)
updating: content/data/arabic/val/state.json (deflated 37%)
updating: content/data/arabic/val/data-00000-of-00001.arrow (deflated 95%)
updating: content/data/arabic/val/dataset_info.json (deflated 72%)
updating: content/data/arabic/train/ (stored 0%)
updating: content/data/arabic/train/state.json (deflated 37%)
updating: content/data/arabic/train/data-00000-of-00001.arrow (deflated 95%)
updating: content/data/arabic/train/dataset_info.json (deflated 72%)
updating: content/data/arabic/.ipynb_checkpoints/ (stored 0%)
updating: content/data/context/ (stored 0%)
updating: content/data/context/val/ (stored 0%)
updating: content/data/context/val/state.json (deflated 37%)
updating: content/data/context/val/data-00000-of-00001.arrow (deflated 75%)
updating: content/data/context/val/dataset_info.json (deflated 70%)
updating: content/data/context/train/ (stored 0%)
updati

In [ ]:
from google.colab import files
files.download("/content/data.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
 model_ar = trainer(tr_ar, tokenizer,num_epochs = 10)


In [ ]:
model_te = trainer(tr_te, tokenizer,num_epochs = 10)

In [ ]:
model_con = trainer(tr_con, tokenizer,num_epochs= 10)

In [ ]:
models = [model_ko, model_ar, model_te, model_con]
training_sets = [tr_ko, tr_ar, tr_te, tr_con]
val_sets = [val_ko, val_ar, val_te, val_con]
topics = ["ko", "ar", "te", "con"]
model_dict = dict(zip(topics, models,training_sets, val_sets))

In [ ]:
model_dict

In [ ]:
import math, torch, torch.nn as nn
from torch.utils.data import DataLoader

def evaluate_perplexity(model, df_val, tokenizer, batch_size=64):
    # df_val: Hugging Face Dataset with "input_ids"
    df_val.set_format("torch", columns=["input_ids"])
    loader = DataLoader(df_val, batch_size=batch_size, shuffle=False)

    device = next(model.parameters()).device
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
    crit = nn.CrossEntropyLoss(ignore_index=pad_id, reduction="sum")

    model.eval()
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device).long()
            logits, _, _ = model(input_ids)                    # [B,T,V]
            # next-token targets (shift left), ignore pads
            loss = crit(
                logits[:, :-1, :].reshape(-1, logits.size(-1)),
                input_ids[:, 1:].reshape(-1)
            )
            total_loss  += loss.item()
            total_tokens += (input_ids[:, 1:] != pad_id).sum().item()

    mean_nll = total_loss / max(1, total_tokens)
    return math.exp(mean_nll)  # perplexity


In [ ]:
model = trainer(train_enc, tokenizer, num_epochs=3, use_labels=False)  # your training fn
val_ppl = evaluate_perplexity(model, val_enc, tokenizer)
print("Validation perplexity:", f"{val_ppl:.2f}")


### Old staff below

In [ ]:
import torch.nn as nn

cos = nn.CosineSimilarity(dim=1, eps=1e-6)
train_tr["cosine_sim"] = cos(
    torch.stack(train_tr["question_embedding"].values),
    torch.stack(train_tr["context_embedding"].values)
).cpu().numpy()


TypeError: stack(): argument 'tensors' (position 1) must be tuple of Tensors, not numpy.ndarray

In [ ]:
!pip install -q scikit-learn


[notice] A new release of pip available: 22.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from sklearn.model_selection import train_test_split
import torch
import pandas as pd

y_all = pd.get_dummies(train_tr["answerable"]).values   # shape (N, num_classes)

x_all = torch.stack(train_tr["context_embedding"].values).numpy()  # (N, embed_dim)

X_train, X_test, y_train, y_test = train_test_split(
    x_all, y_all, test_size=0.1, random_state=42, stratify=y_all
)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)

# CrossEntropyLoss needs integer class labels (not one-hot)
y_train_torch = torch.tensor(y_train.argmax(axis=1), dtype=torch.long)
y_test_torch  = torch.tensor(y_test.argmax(axis=1), dtype=torch.long)


TypeError: stack(): argument 'tensors' (position 1) must be tuple of Tensors, not numpy.ndarray

In [ ]:
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import roc_auc_score


class SimpleClassifier(nn.Module):
    def __init__(self, input_dim, n_classes):
        super(SimpleClassifier, self).__init__()
        self.pool = nn.AdaptiveAvgPool1d(1)   # GlobalAveragePooling1D
        self.fc = nn.Linear(input_dim, n_classes)

    def forward(self, x):
        # x shape: (batch, seq_len, embed_dim)
        x = x.permute(0, 2, 1)        # (batch, embed_dim, seq_len)
        x = self.pool(x).squeeze(-1)  # (batch, embed_dim)
        x = self.fc(x)                # (batch, n_classes)
        return x


NameError: name 'y_train' is not defined

In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_CLASSES = y_train.shape[1]  # assuming one-hot labels
input_dim = x_train.shape[2]  # embedding dimension


TypeError: stack(): argument 'tensors' (position 1) must be tuple of Tensors, not numpy.ndarray

In [ ]:
model = SimpleClassifier(input_dim, N_CLASSES).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# --- Data prep ---
# Convert numpy arrays to torch tensors
X_train = torch.tensor(x_train, dtype=torch.float32)
y_train_torch = torch.tensor(y_train.argmax(axis=1), dtype=torch.long)  # CrossEntropy expects class indices
X_test = torch.tensor(x_test, dtype=torch.float32)
y_test_torch = torch.tensor(y_test.argmax(axis=1), dtype=torch.long)

# --- Training loop ---
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train_torch)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

for epoch in range(10):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * xb.size(0)
        _, preds = outputs.max(1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)

    acc = correct / total
    print(f"Epoch {epoch+1}, Loss: {running_loss/total:.4f}, Acc: {acc:.4f}")

# --- Evaluation (AUC) ---
model.eval()
with torch.no_grad():
    probs = torch.softmax(model(X_test.to(device)), dim=1).cpu().numpy()

auc = roc_auc_score(y_test, probs[:,1])
print("Test AUC:", auc)

In [ ]:
# train_tr["question_translated_tokenized"]
# train_tr["context_tokenized"]

0       [▁The, ▁conflict, ▁, between, ▁France, ▁and, ▁...
1       [▁X, -, rays, ▁make, ▁up, ▁X, -, radiation, ,,...
2       [▁In, ▁2022, ,, ▁Beijing, ▁will, ▁, become, ▁t...
3       [▁The, ▁, British, ▁, Broadcast, ing, ▁Corpora...
4       [▁, Palestine, ▁(, ▁, '),, ▁official, ly, ▁the...
                              ...                        
6330    [▁In, ▁, February, ▁2012, ,, ▁Somali, ▁governm...
6331    [▁The, ▁first, ▁earth, ▁, tracks, ▁were, ▁, cr...
6332    [▁Abdel, ▁Fat, tah, ▁Sa, e, ed, ▁Hu, ssein, ▁K...
6333    [▁, Munich, ▁(, ;, ▁, ;, ▁, ), ▁is, ▁the, ▁cap...
6334    [▁Ana, ba, pt, ism, ▁(, from, ▁Neo, -, Latin, ...
Name: context_tokenized, Length: 6335, dtype: object

In [ ]:
train_tr[["context", "answerable", "question_translated"]][train_tr["answerable"]== False]

,context,answerable,question_translated
17,Moonlight consists of mostly sunlight (with li...,False,Does the moon glow by itself?
36,With the end of the Macedonian Wars – which ra...,False,Was Rome victorious in the Phoenician War?
109,"When on land, an American alligator moves eith...",False,Can crocodiles live on both land and water?
115,Many species produce metabolites that are majo...,False,Can mushrooms heal wounds?
119,The University of Oxford has no known foundati...,False,Who is the founder of Oxford University?
...,...,...,...
6321,Vaccines against anthrax for use in livestock ...,False,In what country was the antidote to tuberculos...
6322,are broken Other Names of the Qur'an: It is be...,False,Who wrote the Qur'an in which Arabic language?
6323,Austin is the capital of the US state of Texas...,False,What is the largest man-made structure in the ...
6324,It became the 'Dravidakajagam' party under Nay...,False,Who was the first Chief Minister of Tamil Nadu?


In [ ]:

# train_tr["contextMquestion"] = train_tr.apply(lambda x: x.columns, axis)

train_tr["contextMquestion"] = train_tr.apply(lambda x: list(set(map(str.lower,x["question_translated_tokenized"]))&set(map(str.lower,x["context_tokenized"]))), axis = 1)

In [ ]:
train_tr[["contextMquestion", "answerable", "question_translated"]][train_tr["answerable"]== False]

,contextMquestion,answerable,question_translated
17,"[▁, ▁moon, ▁the]",False,Does the moon glow by itself?
36,"[▁rome, ▁in, ▁victori, ▁war, ▁the]",False,Was Rome victorious in the Phoenician War?
109,"[▁water, ▁on, ▁land, ▁and, ▁can]",False,Can crocodiles live on both land and water?
115,"[s, ▁m, ushroom, ▁can]",False,Can mushrooms heal wounds?
119,"[▁is, ▁university, ▁of, ▁oxford, ▁the]",False,Who is the founder of Oxford University?
...,...,...,...
6321,"[▁in, ▁, ▁was, ▁the]",False,In what country was the antidote to tuberculos...
6322,"[▁who, an, ▁in, ▁wrote, ', ▁, ▁the, ▁qur, arabic]",False,Who wrote the Qur'an in which Arabic language?
6323,"[▁is, -, ▁in, ▁of, ▁state, ▁, largest, ▁the, ▁...",False,What is the largest man-made structure in the ...
6324,"[▁minister, ▁who, ▁of, chief, ▁was, ▁, ▁the]",False,Who was the first Chief Minister of Tamil Nadu?


In [ ]:
train_tr["question_translated_tokenized"]

0       [▁Who, ▁is, ▁the, ▁w, inner, ▁of, ▁the, ▁Th, i...
1                   [▁Who, ▁discovered, ▁X, -, ray, s, ?]
2       [▁When, ▁was, ▁the, ▁last, ▁Olympic, ▁Games, ▁...
3       [▁What, ', s, ▁the, ▁old, est, ▁broad, caster,...
4       [▁Where, ', s, ▁the, ▁Palest, inian, ▁capital, ?]
                              ...                        
6330    [▁When, ▁did, ▁Somalia, ▁make, ▁its, ▁second, ...
6331    [▁What, ▁was, ▁the, ▁world, ', s, ▁first, ▁tra...
6332    [▁Who, ▁is, ▁Egypt, ', s, ▁leader, ▁in, ▁2019, ?]
6333    [▁What, ▁is, ▁the, ▁most, ▁dens, ely, ▁pop, ul...
6334    [▁Does, ▁Reform, ed, ▁Bapt, ism, ▁have, ▁anyth...
Name: question_translated_tokenized, Length: 6335, dtype: object

In [ ]:
#!pip install -q collections

In [ ]:
type(train_tr['context_tokenized'][0][0])

str

In [ ]:
from collections import Counter

# Flatten both columns into one list of tokens
all_tokens = (
    train_tr['context_tokenized'].dropna().sum()
    + train_tr['question_translated_tokenized'].dropna().sum()
)

# Count token frequencies
token_counts = Counter(all_tokens)


[('▁', 117080), ('▁the', 48219), (',', 36479), ('▁of', 28205), ('.', 27649), ('s', 22796), ('▁and', 20146), ('▁in', 19208), ('a', 14249), ('▁to', 11641), ('▁is', 11447), ('ed', 9936), ('▁was', 8040), ('▁The', 7125), ('?', 6328), ('-', 6322), ('▁(', 6042), ('▁as', 5948), ("'", 5564), ('▁"', 5452), ('▁by', 5408), ('ly', 4649), ('e', 4152), ('ing', 4025), ('▁for', 4024), ('▁with', 3997), ('d', 3888), ('▁on', 3834), ('▁from', 3337), ('▁are', 3181), ('y', 3126), (')', 3093), ('"', 2954), ('▁that', 2919), ('▁first', 2553), ('▁an', 2539), ('which', 2281), ('▁or', 2274), ('▁at', 2263), ('▁be', 2167)]


In [ ]:


# Get top N most frequent tokens
n = 50
top_tokens = token_counts.most_common(n)
top_tokens = list(set([word.lower() for word,_ in top_tokens]))

print(top_tokens)  # list of (token, count) tuples

['),', ')', 'ly', 'e', '▁on', 'y', '▁first', '▁(', 'ing', '▁at', '"', '▁', '▁to', '▁his', 'which', '▁be', "'", '▁"', '▁that', 'd', '.', '▁as', '▁from', '▁with', '▁is', '?', 'ed', '▁by', '▁has', '▁the', '▁and', ',', 'es', 'a', '▁for', '-', '▁in', '▁its', '▁of', 's', '▁are', '▁an', ';', '▁was', '▁it', '▁or', '▁what']


In [ ]:
train_tr[["contextMquestion", "answerable", "question_translated"]][train_tr["answerable"]== False]

,contextMquestion,answerable,question_translated
17,[the],False,Does the moon glow by itself?
36,"[in, war, rome, the]",False,Was Rome victorious in the Phoenician War?
109,"[can, land, on, and, water]",False,Can crocodiles live on both land and water?
115,[can],False,Can mushrooms heal wounds?
119,"[of, university, is, the, oxford]",False,Who is the founder of Oxford University?
...,...,...,...
6321,"[in, was, the]",False,In what country was the antidote to tuberculos...
6322,"[wrote, who, the, in, qur'an, arabic]",False,Who wrote the Qur'an in which Arabic language?
6323,"[of, state, is, in, the, texas, largest]",False,What is the largest man-made structure in the ...
6324,"[was, of, who, the, chief, minister]",False,Who was the first Chief Minister of Tamil Nadu?


In [ ]:
train_tr["conMquest_stripped"] = train_tr["contextMquestion"].apply(lambda wordlist: [word for word in wordlist if word not in top_tokens])

In [ ]:
train_tr[["conMquest_stripped", "answerable", "question_translated"]][train_tr["answerable"]== False]


,conMquest_stripped,answerable,question_translated
17,[▁moon],False,Does the moon glow by itself?
36,"[▁rome, ▁victori, ▁war]",False,Was Rome victorious in the Phoenician War?
109,"[▁water, ▁land, ▁can]",False,Can crocodiles live on both land and water?
115,"[▁m, ushroom, ▁can]",False,Can mushrooms heal wounds?
119,"[▁university, ▁oxford]",False,Who is the founder of Oxford University?
...,...,...,...
6321,[],False,In what country was the antidote to tuberculos...
6322,"[▁who, an, ▁wrote, ▁qur, arabic]",False,Who wrote the Qur'an in which Arabic language?
6323,"[▁state, largest, ▁texas]",False,What is the largest man-made structure in the ...
6324,"[▁minister, ▁who, chief]",False,Who was the first Chief Minister of Tamil Nadu?


In [ ]:
train_tr[["conMquest_stripped", "answerable", "question_translated"]][train_tr["answerable"]== True]

,conMquest_stripped,answerable,question_translated
0,[▁war],True,Who is the winner of the Thirty Years' War?
1,"[▁who, rays, ▁x, ▁discover]",True,Who discovered X-rays?
2,"[▁athens, olympic, ▁games, ▁held]",True,When was the last Olympic Games held in Athens?
3,"[est, ▁old, ▁world, broadcast, er]",True,What's the oldest broadcaster in the world?
4,[▁capital],True,Where's the Palestinian capital?
...,...,...,...
6329,"[▁transit, ▁seoul]",True,When did you start Seoul Transit?
6330,[tion],True,When did Somalia make its second inauguration?
6331,[],True,What was the world's first transportation system?
6332,"[▁who, ▁egypt]",True,Who is Egypt's leader in 2019?


In [ ]:
train_tr["contextMquestion"].apply(lambda x: len(x))


0       3
1       3
2       6
3       5
4       2
       ..
6330    0
6331    3
6332    3
6333    8
6334    2
Name: contextMquestion, Length: 6335, dtype: int64

In [ ]:
import pandas as pd


# Compute average lengths
avg_context_len = train_tr[train_tr["answerable"] == True]["conMquest_stripped"].apply(lambda x: len(x)).mean()
avg_question_len = train_tr[train_tr["answerable"] == False]["conMquest_stripped"].apply(lambda x: len(x)).mean()

print("Average contextMquestion length:", avg_context_len)
print("Average contextMquestion length False:", avg_question_len)


Average contextMquestion length: 2.5025117213663766
Average contextMquestion length False: 2.5234159779614327


In [ ]:
#["question_stripped"])
train_tr["context_stripped"] = train_tr["context_stripped"]

0       ['The', 'conflict', 'between', 'France', 'and'...
1       ['X-rays', 'make', 'up', 'X-radiation', 'a', '...
2       ['In', '2022', 'Beijing', 'will', 'become', 't...
3       ['The', 'British', 'Broadcasting', 'Corporatio...
4       ['Palestine', '', '', 'officially', 'the', 'St...
                              ...                        
6330    ['In', 'February', '2012', 'Somali', 'governme...
6331    ['The', 'first', 'earth', 'tracks', 'were', 'c...
6332    ['Abdel', 'Fattah', 'Saeed', 'Hussein', 'Khali...
6333    ['Munich', '', '', '', 'is', 'the', 'capital',...
6334    ['Anabaptism', 'from', 'Neo-Latin', 'anabaptis...
Name: context_stripped, Length: 6335, dtype: object

In [ ]:
train_tr[["contextMquestion", "answerable", "question_translated"]][train_tr["answerable"]== False]

,contextMquestion,answerable,question_translated
17,"list[{']', '[', ""'"", ' ', ','}]",False,Does the moon glow by itself?
36,"list[{']', '[', ""'"", ' ', ','}]",False,Was Rome victorious in the Phoenician War?
109,"list[{']', '[', ""'"", ' ', ','}]",False,Can crocodiles live on both land and water?
115,"list[{']', '[', ""'"", ' ', ','}]",False,Can mushrooms heal wounds?
119,"list[{']', '[', ""'"", ' ', ','}]",False,Who is the founder of Oxford University?
...,...,...,...
6321,"list[{']', '[', ""'"", ' ', ','}]",False,In what country was the antidote to tuberculos...
6322,"list[{']', '[', ""'"", ' ', ','}]",False,Who wrote the Qur'an in which Arabic language?
6323,"list[{']', '[', ""'"", ' ', ','}]",False,What is the largest man-made structure in the ...
6324,"list[{']', '[', ""'"", ' ', ','}]",False,Who was the first Chief Minister of Tamil Nadu?


In [ ]:
from transformers import AutoTokenizer

model_name = "facebook/nllb-200-distilled-600M"

# Initialize the (fast) tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

# --- Basics ---------------------------------------------------------------

text = "Hello, how are you?"
# enc = tokenizer(text)  # plain dict with 'input_ids' and 'attention_mask'
# print(enc.keys(), enc["input_ids"][:10])

# # With common options
enc = tokenizer(
    text,
    padding="max_length",  # or True / "longest"
    truncation=True,
    max_length=64,
    return_tensors="pt"    # "pt" | "tf" | "np"
)
print(enc.input_ids.shape, enc.attention_mask.shape)
#
# # Batch encode
# batch = tokenizer(
#     ["Hello!", "This is a longer sentence."],
#     padding=True,
#     truncation=True,
#     return_tensors="pt"
# )
#
# # Decode ids back to text
decoded = tokenizer.decode(enc["input_ids"][0], skip_special_tokens=True)
print("decoded", decoded)

ModuleNotFoundError: No module named 'spacy'

In [ ]:

train_tr.to_csv("data/train_translated_tokenized.csv")

In [ ]:
train_tr

,Unnamed: 0,question,context,lang,answerable,answer_start,answer,answer_inlang,question_stripped,question_wordcount,context_stripped,context_wordcount,question_translated,question_translated_stripped,question_translated_wordcount,contextMquestion,conMquest_stripped,question_translated_tokenized,context_tokenized
0,4792,30년 전쟁의 승자는 누구인가?,The conflict between France and Spain continue...,ko,True,21,France,NaN,"[30년, 전쟁의, 승자는, 누구인가]",4,"[The, conflict, between, France, and, Spain, c...",108,Who is the winner of the Thirty Years' War?,"[Who, is, the, winner, of, the, Thirty, Years,...",9,"[war, of, the]",[war],"[▁Who, ▁is, ▁the, ▁w, inner, ▁of, ▁the, ▁Th, i...","[▁The, ▁conflict, ▁between, ▁France, ▁and, ▁Sp..."
1,4793,엑스선은 누가 발견하였는가?,"X-rays make up X-radiation, a form of electrom...",ko,True,503,Wilhelm Röntgen,NaN,"[엑스선은, 누가, 발견하였는가]",3,"[X-rays, make, up, X-radiation, a, form, of, e...",122,Who discovered X-rays?,"[Who, discovered, X-rays]",3,"[discovered, who, x-rays]","[discovered, x-rays]","[▁Who, ▁discovered, ▁X, -, ray, s, ?]","[▁X, -, ray, s, ▁make, ▁up, ▁X, -, radi, ation..."
2,4794,아테네에서 언제 가장 최근의 올림픽이 올렸나요?,"In 2022, Beijing will become the first-ever ci...",ko,True,188,2004,NaN,"[아테네에서, 언제, 가장, 최근의, 올림픽이, 올렸나요]",6,"[In, 2022, Beijing, will, become, the, first-e...",197,When was the last Olympic Games held in Athens?,"[When, was, the, last, Olympic, Games, held, i...",9,"[athens, in, the, olympic, held, games]","[athens, olympic, held, games]","[▁When, ▁was, ▁the, ▁last, ▁Olympic, ▁Games, ▁...","[▁In, ▁202, 2,, ▁Beijing, ▁will, ▁become, ▁the..."
3,4795,세상에서 가장 오래된 방송사는 무엇인가?,The British Broadcasting Corporation (BBC) is ...,ko,True,4,British Broadcasting Corporation (BBC),NaN,"[세상에서, 가장, 오래된, 방송사는, 무엇인가]",5,"[The, British, Broadcasting, Corporation, BBC,...",70,What's the oldest broadcaster in the world?,"[What's, the, oldest, broadcaster, in, the, wo...",7,"[broadcaster, the, world, in, oldest]","[broadcaster, world, oldest]","[▁What, ', s, ▁the, ▁old, est, ▁broad, caster,...","[▁The, ▁British, ▁Broad, casting, ▁Corporation..."
4,4796,팔레스타인 수도는 어딘가요?,"Palestine ( '), officially the State of Palest...",ko,True,205,Jerusalem,NaN,"[팔레스타인, 수도는, 어딘가요]",3,"[Palestine, , , officially, the, State, of, Pa...",84,Where's the Palestinian capital?,"[Where's, the, Palestinian, capital]",4,"[capital, the]",[capital],"[▁Where, ', s, ▁the, ▁Palest, inian, ▁capital, ?]","[▁Palestine, ▁(, ▁', ),, ▁offici, ally, ▁the, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6330,15338,소말리아는 2차 개헌을 언제 했나요?,"In February 2012, Somali government officials ...",ko,True,923,23 June 2012,NaN,"[소말리아는, 2차, 개헌을, 언제, 했나요]",5,"[In, February, 2012, Somali, government, offic...",181,When did Somalia make its second inauguration?,"[When, did, Somalia, make, its, second, inaugu...",7,[],[],"[▁When, ▁did, ▁Somalia, ▁make, ▁its, ▁second, ...","[▁In, ▁February, ▁2012,, ▁Somali, ▁government,..."
6331,15339,세상에서 가장 먼저 시작된 교통수단은 무엇인가?,The first earth tracks were created by humans ...,ko,True,160,animals,NaN,"[세상에서, 가장, 먼저, 시작된, 교통수단은, 무엇인가]",6,"[The, first, earth, tracks, were, created, by,...",118,What was the world's first transportation system?,"[What, was, the, world's, first, transportatio...",7,"[first, was, the]",[],"[▁What, ▁was, ▁the, ▁world, ', s, ▁first, ▁tra...","[▁The, ▁first, ▁earth, ▁tra, cks, ▁were, ▁crea..."
6332,15340,2019년 이집트의 지도자는 누구인가?,"Abdel Fattah Saeed Hussein Khalil El-Sisi ( """"...",ko,True,0,Abdel Fattah Saeed Hussein Khalil El-Sisi,NaN,"[2019년, 이집트의, 지도자는, 누구인가]",4,"[Abdel, Fattah, Saeed, Hussein, Khalil, El-Sis...",30,Who is Egypt's leader in 2019?,"[Who, is, Egypt's, leader, in, 2019]",6,"[is, who, in]",[],"[▁Who, ▁is, ▁Egypt, ', s, ▁leader, ▁in, ▁2019, ?]","[▁Ab, del, ▁Fat, tah, ▁Sae, ed, ▁Hussein, ▁Kha..."
6333,15341,독일에서 가장 인구밀도가 높은 도시는 무엇인가?,Munich (; ; ) is the capital and most populous...,ko,True,205,Berlin,NaN,"[독일에서, 가장, 인구밀도가, 높은, 도시는, 무엇인가]",6,"[Munich, , , , is, the, capital, and, most, 

In [ ]:
import spacy
nlp = spacy.load('en')

def vbd(token):
   """a bad conjugation function"""
   if token.pos_ == 'VERB':
       return token.lemma_ + 'ed'
spacy.tokens.Token.set_extension('vbd', getter=vbd, default=None)
doc = nlp(u'Apple is looking at buying U.K. startup for $1 billion')
for token in doc:
     print(token.text, ":", token._.vbd)

In [ ]:
def count_translated_quest_tokens(df, topn: int = 5): #irrelevant
    langs = ["ko", "ar", "te"]
    col  = "question_translated"
    for l in langs:
        mask = df["lang"] == l
        lang_df = df.loc[mask].copy()
        if lang_df[col].dtype == object and lang_df[col].map(lambda x: isinstance(x, list)).any():
            tokens = list(chain.from_iterable(x or [] for x in lang_df[col].tolist()))
        else:
            tokens = list(chain.from_iterable((str(x or "")).split() for x in lang_df[col].tolist()))
        tokens = [t.strip() for t in tokens if isinstance(t, str) and t.strip()]
        counts = pd.Series(tokens).value_counts().head(topn)

        out = counts.reset_index()
        out.columns = ["token", "count"]
        return out #


In [ ]:
train_tr[train_tr["answerable"] == True]

,Unnamed: 0,question,context,lang,answerable,answer_start,answer,answer_inlang,question_stripped,question_wordcount,context_stripped,context_wordcount,question_translated
0,4792,30년 전쟁의 승자는 누구인가?,The conflict between France and Spain continue...,ko,True,21,France,NaN,"['30년', '전쟁의', '승자는', '누구인가']",4,"['The', 'conflict', 'between', 'France', 'and'...",108,Who is the winner of the Thirty Years' War?
1,4793,엑스선은 누가 발견하였는가?,"X-rays make up X-radiation, a form of electrom...",ko,True,503,Wilhelm Röntgen,NaN,"['엑스선은', '누가', '발견하였는가']",3,"['X-rays', 'make', 'up', 'X-radiation', 'a', '...",122,Who discovered X-rays?
2,4794,아테네에서 언제 가장 최근의 올림픽이 올렸나요?,"In 2022, Beijing will become the first-ever ci...",ko,True,188,2004,NaN,"['아테네에서', '언제', '가장', '최근의', '올림픽이', '올렸나요']",6,"['In', '2022', 'Beijing', 'will', 'become', 't...",197,When was the last Olympic Games held in Athens?
3,4795,세상에서 가장 오래된 방송사는 무엇인가?,The British Broadcasting Corporation (BBC) is ...,ko,True,4,British Broadcasting Corporation (BBC),NaN,"['세상에서', '가장', '오래된', '방송사는', '무엇인가']",5,"['The', 'British', 'Broadcasting', 'Corporatio...",70,What's the oldest broadcaster in the world?
4,4796,팔레스타인 수도는 어딘가요?,"Palestine ( '), officially the State of Palest...",ko,True,205,Jerusalem,NaN,"['팔레스타인', '수도는', '어딘가요']",3,"['Palestine', '', '', 'officially', 'the', 'St...",84,Where's the Palestinian capital?
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6329,15337,서울교통공사는 언제 창단했나요?,Seoul Metro (Hangul: 서울메트로) is a public corpor...,ko,True,111,1970,NaN,"['서울교통공사는', '언제', '창단했나요']",3,"['Seoul', 'Metro', 'Hangul', '서울메트로', 'is', 'a...",47,When did you start Seoul Transit?
6330,15338,소말리아는 2차 개헌을 언제 했나요?,"In February 2012, Somali government officials ...",ko,True,923,23 June 2012,NaN,"['소말리아는', '2차', '개헌을', '언제', '했나요']",5,"['In', 'February', '2012', 'Somali', 'governme...",181,When did Somalia make its second inauguration?
6331,15339,세상에서 가장 먼저 시작된 교통수단은 무엇인가?,The first earth tracks were created by humans ...,ko,True,160,animals,NaN,"['세상에서', '가장', '먼저', '시작된', '교통수단은', '무엇인가']",6,"['The', 'first', 'earth', 'tracks', 'were', 'c...",118,What was the world's first transportation system?
6332,15340,2019년 이집트의 지도자는 누구인가?,"Abdel Fattah Saeed Hussein Khalil El-Sisi ( """"...",ko,True,0,Abdel Fattah Saeed Hussein Khalil El-Sisi,NaN,"['2019년', '이집트의', '지도자는', '누구인가']",4,"['Abdel', 'Fattah', 'Saeed', 'Hussein', 'Khali...",30,Who is Egypt's leader in 2019?


In [ ]:
train_tr[train_tr["answerable"] == False]

,Unnamed: 0,question,context,lang,answerable,answer_start,answer,answer_inlang,question_stripped,question_wordcount,context_stripped,context_wordcount,question_translated
17,4809,달은 자체발광하는가?,Moonlight consists of mostly sunlight (with li...,ko,False,-1,no,NaN,"['달은', '자체발광하는가']",2,"['Moonlight', 'consists', 'of', 'mostly', 'sun...",21,Does the moon glow by itself?
36,4828,포에니 전쟁에서 로마가 승리했나요?,With the end of the Macedonian Wars – which ra...,ko,False,-1,no,NaN,"['포에니', '전쟁에서', '로마가', '승리했나요']",4,"['With', 'the', 'end', 'of', 'the', 'Macedonia...",78,Was Rome victorious in the Phoenician War?
109,4901,악어는 땅과 물 둘 다에서 살 수 있을까?,"When on land, an American alligator moves eith...",ko,False,-1,no,NaN,"['악어는', '땅과', '물', '둘', '다에서', '살', '수', '있을까']",8,"['When', 'on', 'land', 'an', 'American', 'alli...",147,Can crocodiles live on both land and water?
115,4907,곰팡이가 상처를 치료할 수 있을까?,Many species produce metabolites that are majo...,ko,False,-1,no,NaN,"['곰팡이가', '상처를', '치료할', '수', '있을까']",5,"['Many', 'species', 'produce', 'metabolites', ...",301,Can mushrooms heal wounds?
119,4911,옥스퍼드 대학의 창립주는 누구인가요?,The University of Oxford has no known foundati...,ko,False,-1,no,NaN,"['옥스퍼드', '대학의', '창립주는', '누구인가요']",4,"['The', 'University', 'of', 'Oxford', 'has', '...",109,Who is the founder of Oxford University?
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6321,15322,క్షయ వ్యాధికి విరుగుడు ఏ దేశంలో కనుగొన్నారు?,Vaccines against anthrax for use in livestock ...,te,False,-1,France,ఫ్రాన్స్,"['క్షయ', 'వ్యాధికి', 'విరుగుడు', 'ఏ', 'దేశంలో'...",6,"['Vaccines', 'against', 'anthrax', 'for', 'use...",100,In what country was the antidote to tuberculos...
6322,15323,ఖురాన్ ఏ అరబ్బీ భాషలో ఎవరు రాసారు?,are broken Other Names of the Qur'an: It is be...,te,False,-1,Prophet Muhammad,ముహమ్మద్ ప్రవక్త,"['ఖురాన్', 'ఏ', 'అరబ్బీ', 'భాషలో', 'ఎవరు', 'రా...",6,"['are', 'broken', 'Other', 'Names', 'of', 'the...",139,Who wrote the Qur'an in which Arabic language?
6323,15324,టెక్సస్ రాష్ట్రంలోని అతిపెద్ద మానవ నిర్మితం ఏది ?,Austin is the capital of the US state of Texas...,te,False,-1,JP Morgan Chase Tower,జేపీ మోర్గాన్ ఛేజ్ టవర్,"['టెక్సస్', 'రాష్ట్రంలోని', 'అతిపెద్ద', 'మానవ'...",7,"['Austin', 'is', 'the', 'capital', 'of', 'the'...",135,What is the largest man-made structure in the ...
6324,15325,తమిళనాడులో రాష్ట్ర మొదటి ముఖ్యమంత్రి ఎవరు?,It became the 'Dravidakajagam' party under Nay...,te,False,-1,C. N. Annadurai,సి.ఎన్.అన్నాదురై,"['తమిళనాడులో', 'రాష్ట్ర', 'మొదటి', 'ముఖ్యమంత్ర...",5,"['It', 'became', 'the', 'Dravidakajagam', 'par...",143,Who was the first Chief Minister of Tamil Nadu?


### Idea: look at cosine difference between question and answer, but it will very likely fail. Other idea is too look at how questions are formed and if it is possible to extract certain information for the context. My theory that certain types of starting tockens correspond to non-answerable questions.

In [ ]:
train_set

In [ ]:
train_ds

## LSTM language model


In [34]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Current device index:", torch.cuda.current_device())
    print("Current device name:", torch.cuda.get_device_name(torch.cuda.current_device()))

# Choose a device handle for your code
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using DEVICE =", DEVICE)


CUDA available: False
CUDA device count: 0
Using DEVICE = cpu


In [37]:
import torch, platform
print("is_available:", torch.cuda.is_available())
print("torch.version.cuda:", torch.version.cuda)
print("compiled w/ cudnn:", torch.backends.cudnn.is_available())
print("torch.__version__:", torch.__version__)
print("platform:", platform.platform())


is_available: False
torch.version.cuda: None
compiled w/ cudnn: False
torch.__version__: 2.8.0+cpu
platform: Windows-10-10.0.26100-SP0


In [32]:
train = pd.read_csv("data/train_translated.csv")
validation = pd.read_csv("data/validation_translated.csv")

In [29]:
train.head()

,Unnamed: 0,question,context,lang,answerable,answer_start,answer,answer_inlang,question_stripped,question_wordcount,context_stripped,context_wordcount,question_translated
0,4792,30년 전쟁의 승자는 누구인가?,The conflict between France and Spain continue...,ko,True,21,France,NaN,"['30년', '전쟁의', '승자는', '누구인가']",4,"['The', 'conflict', 'between', 'France', 'and'...",108,Who is the winner of the Thirty Years' War?
1,4793,엑스선은 누가 발견하였는가?,"X-rays make up X-radiation, a form of electrom...",ko,True,503,Wilhelm Röntgen,NaN,"['엑스선은', '누가', '발견하였는가']",3,"['X-rays', 'make', 'up', 'X-radiation', 'a', '...",122,Who discovered X-rays?
2,4794,아테네에서 언제 가장 최근의 올림픽이 올렸나요?,"In 2022, Beijing will become the first-ever ci...",ko,True,188,2004,NaN,"['아테네에서', '언제', '가장', '최근의', '올림픽이', '올렸나요']",6,"['In', '2022', 'Beijing', 'will', 'become', 't...",197,When was the last Olympic Games held in Athens?
3,4795,세상에서 가장 오래된 방송사는 무엇인가?,The British Broadcasting Corporation (BBC) is ...,ko,True,4,British Broadcasting Corporation (BBC),NaN,"['세상에서', '가장', '오래된', '방송사는', '무엇인가']",5,"['The', 'British', 'Broadcasting', 'Corporatio...",70,What's the oldest broadcaster in the world?
4,4796,팔레스타인 수도는 어딘가요?,"Palestine ( '), officially the State of Palest...",ko,True,205,Jerusalem,NaN,"['팔레스타인', '수도는', '어딘가요']",3,"['Palestine', '', '', 'officially', 'the', 'St...",84,Where's the Palestinian capital?
